# Entity Topic Classification (Batch API)
This notebook mirrors the 4_jsonl pipeline but classifies entities into predefined topics and labels their level as 'higher', 'normal', or 'lower'.

In [1]:
import os
import json
import re
from typing import List, Dict, Any, Iterable, Optional, Literal
import pandas as pd
import tqdm
import openai
from openai import OpenAI

In [2]:
# Load API key (expects this notebook to live under Code/)
with open('_secret_key4', 'r') as f:
    openai_key = f.read()
os.environ['OPENAI_API_KEY'] = openai_key
client = OpenAI()

In [3]:
# Load speeches dataset (same as 4_jsonl)
df = pd.read_csv('../Data/Speeches/all_cb_speeches.csv', sep='\t')
df.head()

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source,date,speech_id
0,https://www.bankofgreece.gr/en/news-and-media/...,NaN,"Opening remarks at the ""GBA Session on Non-Per...",NaN,2019-09-11,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Opening remarks by Dimitris Malliaropulos, Chi...",NaN,grc_dimitris_malliaropulos_11_09_2019_english,English,CB websites,2019-09-11,1
1,https://www.bankofgreece.gr/en/news-and-media/...,NaN,"Opening Remarks at the IFFR Conference: ""Under...",NaN,2019-09-12,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Opening Remarks by Dimitris Malliaropulos, Chi...",NaN,grc_dimitris_malliaropulos_12_09_2019_english,English,CB websites,2019-09-12,2
2,https://www.bankofgreece.gr/enimerosi/grafeio-...,NaN,"Speech at the GetInvolved conference: ""The dec...",NaN,2019-12-13,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Speech by Dimitris Malliaropoulos, Chief Econo...",", & GetInvolved : ...",grc_dimitris_malliaropulos_13_12_2019_greek,Greek,CB websites,2019-12-13,3
3,https://www.bankofgreece.gr/en/news-and-media/...,NaN,Speech at the AHK Europa Konferenz: Remarks on...,NaN,2019-09-20,Dimitris Malliaropulos,Senior management,Male,Bank of Greece,GRC,"Speech by Dimitris Malliaropulos, Chief Econom...",NaN,grc_dimitris_malliaropulos_20_09_2019_english,English,CB websites,2019-09-20,4
4,https://www.bankofgreece.gr/en/news-and-media/...,NaN,"Speech: ""Assessing the performance and regulat...",NaN,2010-02-04,Eleni D Dendrinou-Louri,Deputy Governor,Female,Bank of Greece,GRC,"Speech of the Dep. Governor E. Louri: ""Assessi...",NaN,grc_eleni_d_dendrinou-louri_04_02_2010_english,English,CB websites,2010-02-04,5


In [4]:
df_Mann = pd.read_csv('Catherine_man.csv')
df_Mann

,cause_entity,effect_entity,relationship,document_reference,quantifications,modulator_uncertainty,modulator_temporal,tense,custom_id,speech_id,batch_id,speech_narrative_id
0,monetary policy,short-circuit the inflationary expectations dy...,Monetary policy tools are used to influence in...,The challenge is to deploy monetary policy too...,NaN,NaN,NaN,PRESENT,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_0
1,monetary policy,moderate the hit to purchasing power,Monetary policy tools are used to influence in...,The challenge is to deploy monetary policy too...,NaN,NaN,NaN,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_1
2,current inflation,affecting future price inflation,Current inflation influences price expectation...,Uncertainties about price expectations and pri...,NaN,NaN,NaN,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_2
3,price realisations,evolution of real income and demand,Price realisations affect real income and dema...,"Relatedly, but its own source of uncertainty i...",NaN,NaN,medium-term,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_3
4,monetary policy,influence inflation and output,Monetary policy affects inflation and output t...,"Another uncertainty, particularly important fo...",NaN,NaN,medium term,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_4
...,...,...,...,...,...,...,...,...,...,...,...,...
303,contractionary monetary policy shock,output declines,Tightening monetary policy reduces economic ac...,"After a contractionary monetary policy shock, ...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_54
304,output decline,reduction in the demand for emissions,"Lower output reduces industrial activity, decr...","After a contractionary monetary policy shock, ...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_55
305,output decline under ETS,carbon price falls,Reduced demand for emissions under an ETS lead...,Under an ETS the carbon price falls (while und...,NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_56
306,fall in carbon price under ETS,boosts output in foreign country,"Lower carbon prices make polluting cheaper, en...","In the foreign country, the fall in price unde...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_57


In [5]:
df_Mann

,cause_entity,effect_entity,relationship,document_reference,quantifications,modulator_uncertainty,modulator_temporal,tense,custom_id,speech_id,batch_id,speech_narrative_id
0,monetary policy,short-circuit the inflationary expectations dy...,Monetary policy tools are used to influence in...,The challenge is to deploy monetary policy too...,NaN,NaN,NaN,PRESENT,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_0
1,monetary policy,moderate the hit to purchasing power,Monetary policy tools are used to influence in...,The challenge is to deploy monetary policy too...,NaN,NaN,NaN,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_1
2,current inflation,affecting future price inflation,Current inflation influences price expectation...,Uncertainties about price expectations and pri...,NaN,NaN,NaN,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_2
3,price realisations,evolution of real income and demand,Price realisations affect real income and dema...,"Relatedly, but its own source of uncertainty i...",NaN,NaN,medium-term,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_3
4,monetary policy,influence inflation and output,Monetary policy affects inflation and output t...,"Another uncertainty, particularly important fo...",NaN,NaN,medium term,FUTURE,speech_16649_batch_0,16649,batch_6956d9714e688190be18ed9b79ec69e7,16649_4
...,...,...,...,...,...,...,...,...,...,...,...,...
303,contractionary monetary policy shock,output declines,Tightening monetary policy reduces economic ac...,"After a contractionary monetary policy shock, ...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_54
304,output decline,reduction in the demand for emissions,"Lower output reduces industrial activity, decr...","After a contractionary monetary policy shock, ...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_55
305,output decline under ETS,carbon price falls,Reduced demand for emissions under an ETS lead...,Under an ETS the carbon price falls (while und...,NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_56
306,fall in carbon price under ETS,boosts output in foreign country,"Lower carbon prices make polluting cheaper, en...","In the foreign country, the fall in price unde...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_6956d97820a48190b7d8da5b21e4b26f,16655_57


In [6]:
# Sentence splitting utilities (adapted from 4_jsonl)
def split_sentences(text: str) -> List[str]:
    exceptions = [
        'Mr', 'Mrs', 'Ms', 'Dr', 'Prof', 'Sr', 'Jr', 'St', 'Mt', 'vs',
        'etc', 'e.g', 'i.e', 'Fig', 'Inc', 'Ltd', 'Co', 'No', 'U.S', 'U.K'
    ]
    pattern = re.compile(r'(?<!\w\.\w)(?<![A-Z][a-z]\.)(?<=\.)\s+(?=[A-Z])')
    candidates = pattern.split(text)
    repaired = []
    for i, part in enumerate(candidates):
        if i == 0:
            repaired.append(part)
        else:
            prev = repaired[-1].strip()
            last_word = prev.split()[-1].rstrip('.')
            if last_word in exceptions or re.match(r'^[A-Z]\.?$', last_word):
                repaired[-1] = prev + ' ' + part
            else:
                repaired.append(part)
    return [s.strip() for s in repaired if s.strip()]

def add_split_sentences(df: pd.DataFrame, text_col: str = 'text') -> pd.DataFrame:
    df['split_sentences'] = df[text_col].apply(split_sentences)
    return df

def create_batches_from_sentences(sentences: List[str], batch_len: Optional[int] = None, pct_of_batch: float = 0.1) -> List[List[str]]:
    if batch_len is None:
        batch_len_calc = max(int(len(sentences) * pct_of_batch), 1)
        return [sentences[i:i + batch_len_calc] for i in range(0, len(sentences), batch_len_calc)]
    else:
        return [sentences[i:i + batch_len] for i in range(0, len(sentences), batch_len)]

def prepare_batches(df: pd.DataFrame, batch_len: Optional[int], text_col: str = 'text', pct_of_batch: float = 0.1) -> List[Dict[str, Any]]:
    df = add_split_sentences(df, text_col)
    output: List[Dict[str, Any]] = []
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc='Processing Rows'):
        batches = create_batches_from_sentences(row['split_sentences'], batch_len, pct_of_batch)
        output.append({'speech_id': row['speech_id'], 'batches': batches})
    return output

In [ ]:
# Define topics (EDIT THIS LIST as needed) with descriptions
TOPICS: List[str] = [
    'inflation',  'output', #'inflation/output tradeoffs',
    'labour market', 'interest rates',
    'government spending',
    'exchange rate', 'housing market',
    'credit', 'financial markets',
    'geopolitical shocks', 'trade', #'central bank communication',
    'productivity', 'other',
]
TOPIC_DESCRIPTIONS: Dict[str, str] = {
    'inflation': 'General price level dynamics (CPI, inflation expectations, price pressures).',
    'output': 'Economic activity and production levels (growth, output gaps).',
    #'inflation/output tradeoffs': 'Policy tradeoffs between stabilizing inflation and supporting output/growth.',
    'labour market': 'Employment, unemployment, wages, participation, hiring conditions.',
    'interest rates': 'Policy rates and market yields (short-term rates, yield curve).',
    'government spending': 'Fiscal expenditure, budgets, and public-sector demand.',
    'exchange rate': 'Currency valuation and FX movements.',
    'housing market': 'Housing activity, construction, and house prices.',
    'credit conditions': 'Lending volumes, credit conditions, spreads, and access to finance.',
    'financial markets': 'Equities, bonds, liquidity, volatility, and risk sentiment.',
    'geopolitical shocks': 'Wars, sanctions, and political disruptions affecting the economy.',
    'trade': 'Exports, imports, trade balance, tariffs, and global demand.',
    'productivity': 'Efficiency, technology, and output per worker or per hour.',
    #'central bank communication': "Central banks' communication to financial markets about the trajectory of policy.",
    'other': 'Entities or topics not covered by the predefined categories.',
}
def build_entity_classification_schema(topics: List[str]) -> Dict[str, Any]:
    if not topics:
        raise ValueError('Topics list must be non-empty.')
    # Build per-topic descriptions using oneOf + const entries
    topic_one_of = [
        {
            'const': t,
            'description': TOPIC_DESCRIPTIONS.get(t, 'Topic not otherwise specified.')
        }
        for t in topics
    ]
    return {
        'name': 'EntityTopicClassificationList',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'items': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'entity': { 'type': 'string', 'description': 'The specific economic entity/variable referenced by the policymaker.' },
                            'topic': {
                                'type': 'string',
                                'enum': topics,
                                'description': 'Closest topic label from the list with per-topic meanings.',
                                'oneOf': topic_one_of,
                            },
                            'classification': {
                                'type': 'string',
                                'enum': ['higher',  'lower'],ss
                                'description': (
                                    "Classification relative to baseline according to the policymaker. For instance 'risk to the upward path of inflation' means the inflation rate might go down relative to baseline."
                                    "higher: an upward shock, risk, increase or higher than baseline value of the economic entity / variable / event described; includes the sudden appearance of the entity according to the speaker. This should be viewed from the standpoint of the policymaker's reaction function."
                                    "lower: a downward shock, risk, decrease or lower than baseline value of the economic entity / variable / event described; includes the sudden disappearance of the entity according to the speaker. This should be viewed from the standpoint of the policymaker's reaction function."
                                )
                            },
                            'location': {
                                'type': 'string',
                                'enum': ['domestic', 'global'],
                                'description': (
                                    "Whether the entity refers to a domestic or global phenomenon relative to the speaker's central bank jurisdiction. "
                                    "domestic: entities specific to the speaker's country/region (e.g., 'UK inflation', 'domestic labor market', country-specific policies). "
                                    "global: entities with international or worldwide scope (e.g., 'global inflation', 'international trade', 'world economy', or entities from other countries/regions)."
                                )
                            },
                            #'document_reference': {
                            #    'type': 'string',
                            #    'description': 'The exact sentence from which the entity/topic/classification is determined.'
                            #}
                        },
                        'required': ['entity', 'topic', 'classification', 'location',
                                      #'document_reference'
                                      ]
                    }
                }
            },
            'required': ['items']
        }
    }

In [ ]:
# Validation model
# Validation model (replace existing cell)
from pydantic import BaseModel, ValidationError
class EntityTopicClassification(BaseModel):
    entity: str
    topic: str
    classification: Literal['higher',  'lower']
    location: Literal['domestic', 'global']
    #document_reference: str

In [ ]:
SYSTEM_INSTRUCTIONS = """
You will receive entities from a central banker's speech along with the speaker's name and central bank. 
You should classify economic entities from speech text.
For each explicit entity/variable mentioned:
1) Assign the closest topic from the provided topics list.
2) Classify the described level/intensity as higher or lower relative to typical baseline using the sentence as context.
3) Classify the location as domestic or global relative to the speaker's central bank jurisdiction.
Return ONLY a JSON object with key 'items' containing the array of objects.
"""

USER_TEMPLATE = """
Speaker: {speaker_name} from {central_bank}

Analyze the following sentences, extract referenced entities/variables,
map each to the closest topic from this list: {topics}.
For each entity:
- Set classification to higher/lower based on the description (e.g., increases=higher, declines=lower).
- Set location to 'domestic' if the entity refers to the speaker's country/jurisdiction, or 'global' if it refers to international/worldwide phenomena or other countries.

You are given the following sentence for context to help you identify the topic, classification, and location.
Sentence:
{sentences}

Output: JSON object only, with key 'items' containing the array.
"""

In [10]:
def _normalize_classification(value: str) -> str:
    v = (value or '').strip().lower()
    return v if v in {'higher', 'normal', 'lower'} else 'normal'

def extract_batch_entity_topics(
    sentences: List[str],
    topics: List[str],
    model: str = 'gpt-4o',
    temperature: float = 0.0
) -> List[Dict[str, str]]:
    schema = build_entity_classification_schema(topics)
    prompt = USER_TEMPLATE.format(
        topics=', '.join(topics),
        sentences='\n'.join(f'- {s}' for s in sentences),
    )
    resp = client.responses.create(
        model=model,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': SYSTEM_INSTRUCTIONS},
            {'role': 'user', 'content': prompt},
        ],
        text={
            'format': {
                'type': 'json_schema',
                'name': schema['name'],
                'schema': schema['schema'],
                'strict': schema.get('strict', True),
            }
        },
    )
    json_text: Optional[str] = None
    if hasattr(resp, 'output_text') and isinstance(resp.output_text, str):
        json_text = resp.output_text
    else:
        try:
            content = resp.output[0].content[0]
            if getattr(content, 'type', None) == 'output_json':
                json_text = content.output_json
            elif getattr(content, 'type', None) == 'text':
                json_text = content.text
        except Exception:
            pass
    if not json_text:
        raise RuntimeError('No textual JSON output returned from the model.')
    try:
        raw = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(f'Model did not return valid JSON: {e}')
    if isinstance(raw, dict) and 'items' in raw and isinstance(raw['items'], list):
        raw_list = raw['items']
    elif isinstance(raw, list):
        raw_list = raw
    else:
        raise ValueError("Expected a JSON object with 'items' array or a JSON array.")
    results: List[Dict[str, str]] = []
    for item in raw_list:
        try:
            obj = EntityTopicClassification(**item)
        except ValidationError as e:
            raise ValueError(f'Invalid item schema: {e}')
        topic_val = (obj.topic or '').strip()
        if topic_val not in topics:
            topic_val = topics[0]
        results.append({
            'entity': (obj.entity or '').strip(),
            'topic': topic_val,
            'classification': _normalize_classification(obj.classification),
            #'document_reference': (obj.document_reference or '').strip(),
        })
    return results

def send_batches_entity_topics(
    prepared_batches: List[Dict[str, Any]],
    topics: List[str],
    model: str = 'gpt-4o',
    temperature: float = 0.0
) -> List[Dict[str, Any]]:
    aggregated: List[Dict[str, Any]] = []
    for row in prepared_batches:
        speech_id = row.get('speech_id')
        batches = row.get('batches', [])
        batch_results: List[List[Dict[str, str]]] = []
        for sentences in batches:
            try:
                parsed = extract_batch_entity_topics(sentences, topics, model=model, temperature=temperature)
            except Exception:
                parsed = []
            batch_results.append(parsed)
        aggregated.append({ 'speech_id': speech_id, 'batch_results': batch_results })
    return aggregated

In [11]:
def _build_jsonl_lines_for_entities_from_csv(
    source_csv_path: str,
    topics: List[str],
    model: str,
    temperature: float,
 ) -> List[str]:
    # Load source CSV
    df_src = pd.read_csv(source_csv_path)
    required = {'cause_entity','effect_entity','speech_id','speech_narrative_id'}
    if not required.issubset(df_src.columns):
        raise ValueError("CSV must contain columns: cause_entity, effect_entity, speech_id, speech_narrative_id.")
    
    # Load speeches dataframe to get speaker info
    df_speeches = pd.read_csv('../Data/Speeches/all_cb_speeches.csv', sep='\t')
    # Create lookup dictionary: speech_id -> (Authorname, CentralBank)
    speaker_info = {}
    for _, row in df_speeches.iterrows():
        sid = row.get('speech_id')
        author = row.get('Authorname', 'Unknown Speaker')
        cb = row.get('CentralBank', 'Unknown Central Bank')
        if pd.notna(sid):
            speaker_info[str(sid)] = (str(author), str(cb))
    
    lines: List[str] = []
    for idx, row in df_src.iterrows():
        speech_id = str(row['speech_id'])
        narr_id = str(row['speech_narrative_id'])
        docref = str(row['document_reference']).strip() if 'document_reference' in df_src.columns and pd.notna(row['document_reference']) else ''
        
        # Get speaker info for this speech
        speaker_name, central_bank = speaker_info.get(speech_id, ('Unknown Speaker', 'Unknown Central Bank'))
        
        for entity_type, col in [('cause','cause_entity'), ('effect','effect_entity')]:
            entity = str(row[col]).strip() if pd.notna(row[col]) else ''
            if not entity:
                continue
            # Compose prompt focusing only on the target entity within its sentence context
            sentences = docref if docref else entity
            focus_line = f"Focus ONLY on entity: {entity}. Return a single item in 'items'."
            prompt = USER_TEMPLATE.format(
                speaker_name=speaker_name,
                central_bank=central_bank,
                topics=', '.join(topics),
                sentences=f"{sentences}\n{focus_line}",
            )
            schema = build_entity_classification_schema(topics)
            body = {
                'model': model,
                'temperature': temperature,
                'input': [
                    {'role': 'system', 'content': SYSTEM_INSTRUCTIONS},
                    {'role': 'user', 'content': prompt},
                ],
                'text': {
                    'format': {
                        'type': 'json_schema',
                        'name': schema['name'],
                        'schema': schema['schema'],
                        'strict': schema.get('strict', True),
                    }
                },
            }
            custom_id = f"speech_{speech_id}_narr_{narr_id}_{entity_type}_{idx}"
            row_obj = {
                'custom_id': custom_id,
                'method': 'POST',
                'url': '/v1/responses',
                'body': body,
            }
            lines.append(json.dumps(row_obj, ensure_ascii=False))
    return lines
def _build_jsonl_lines_for_speech_entity_topics(
    speech_id: int | str,
    batches: Iterable[Iterable[str]],
    topics: List[str],
    model: str,
    temperature: float,
 ) -> List[str]:
    schema = build_entity_classification_schema(topics)
    lines: List[str] = []
    for b_idx, sentences in enumerate(batches):
        prompt = USER_TEMPLATE.format(
            topics=', '.join(topics),
            sentences='\n'.join(f'- {s}' for s in sentences)
        )
        body = {
            'model': model,
            'temperature': temperature,
            'input': [
                {'role': 'system', 'content': SYSTEM_INSTRUCTIONS},
                {'role': 'user', 'content': prompt},
            ],
            'text': {
                'format': {
                    'type': 'json_schema',
                    'name': schema['name'],
                    'schema': schema['schema'],
                    'strict': schema.get('strict', True),
                }
            },
        }
        row = {
            'custom_id': f'speech_{speech_id}_entity_topics_batch_{b_idx}',
            'method': 'POST',
            'url': '/v1/responses',
            'body': body,
        }
        lines.append(json.dumps(row, ensure_ascii=False))
    return lines

def create_speech_batch_and_submit_entity_topics(
    speech_id: int | str | None = None,
    prepared_batches: Optional[List[Dict[str, Any]]] = None,
    topics: List[str] = TOPICS,
    model: str = 'gpt-4o',
    temperature: float = 0.0,
    jsonl_save_path: Optional[str] = None,
    completion_window: str = '24h',
    metadata: Optional[Dict[str, Any]] = None,
    source_csv_path: Optional[str] = None,
 ) -> Dict[str, Any]:
    """
    Create a Batch API job. If `source_csv_path` is provided, build one request per entity
    using CSV columns (cause/effect, speech_id, speech_narrative_id, document_reference).
    Otherwise, fall back to sentence-batch mode using `prepared_batches` and `speech_id`.
    """
    if source_csv_path:
        lines = _build_jsonl_lines_for_entities_from_csv(source_csv_path, topics=topics, model=model, temperature=temperature)
        save_tag = 'entities_from_csv'
    else:
        print("error")
        print(7/0)
        """if prepared_batches is None or speech_id is None:
            raise ValueError("prepared_batches and speech_id are required when source_csv_path is not provided.")
        entry = next((r for r in prepared_batches if r.get('speech_id') == speech_id), None)
        if entry is None:
            raise ValueError(f'speech_id {speech_id} not found in prepared_batches')
        batches = entry.get('batches', [])
        lines = _build_jsonl_lines_for_speech_entity_topics(speech_id, batches, topics, model=model, temperature=temperature)
        save_tag = f'speech_{speech_id}_entity_topics'"""
    if jsonl_save_path is None:
        os.makedirs('user_outputs/speech_runs', exist_ok=True)
        jsonl_save_path = f'user_outputs/speech_runs/{save_tag}.jsonl'
    with open(jsonl_save_path, 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')
    uploaded = client.files.create(file=open(jsonl_save_path, 'rb'), purpose='batch')
    batch_metadata = {'mode': 'entity_topics', 'source_csv_path': source_csv_path or ''}
    if metadata:
        batch_metadata.update(metadata)
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint='/v1/responses',
        completion_window=completion_window,
        metadata=batch_metadata,
    )
    print(f"Batch created. id={batch.id} status={batch.status} input_file_id={uploaded.id} completion_window={completion_window}")
    return {
        'batch_id': batch.id,
        'status': batch.status,
        'input_file_id': uploaded.id,
        'jsonl_path': jsonl_save_path,
        'completion_window': completion_window,
        'metadata': batch.metadata,
    }

In [27]:
def retrieve_batch_output_entity_topics(batch_id: str, source_csv_path: Optional[str] = None) -> pd.DataFrame:
    batch = client.batches.retrieve(batch_id)
    if batch.status != 'completed':
        print(f"Batch {batch_id} not completed yet. Status: {batch.status}")
        return pd.DataFrame()
    out_file_id = batch.output_file_id
    file_stream = client.files.content(out_file_id)
    df_rows: List[Dict[str, Any]] = []
    for raw_line in file_stream.read().decode('utf-8').splitlines():
        obj = json.loads(raw_line)
        custom_id = obj.get('custom_id', '')
        body = obj.get('response', {}).get('body')
        print(body)
        items_list: List[Dict[str, Any]] = []
        #for message in body:
        #    print(message)

        # Handle message-list format
        if isinstance(body, list):
            for message in body:
                content_entries = message.get('content', [])
                print(content_entries)
                for c in content_entries:
                    if c.get('type') == 'output_text':
                        text_json = c.get('text', '{}')
                        try:
                            payload = json.loads(text_json)
                            items = payload.get('items', [])
                            if isinstance(items, list):
                                items_list.extend(items)
                        except json.JSONDecodeError:
                            # Skip unparseable outputs
                            continue
        # Fallback: dict format with 'output' list
        elif isinstance(body, dict):
            outputs = body.get('output', [])
            print(outputs)
            for o in outputs:
                #if o.get('type') == 'output_text':
                try:
                    payload = o.get('content',[])[0]
                    payload = payload.get('text', '{}')
                    payload = payload.replace('```json','')
                    payload = payload.replace('```','')
                    payload = json.loads(payload)
                    items = payload.get('items', [])
                    if isinstance(items, list):
                        items_list.extend(items)
                except json.JSONDecodeError:
                    print('error in json',o.get('content',[])[0].get('text', '{}'))
                    continue
        # Parse custom_id for speech_id, narrative, entity_type, and idx
        speech_id = None
        speech_narrative_id = None
        entity_type = None
        idx = None
        m = re.match(r"speech_(?P<sid>[^_]+)_narr_(?P<narr>.*?)_(?P<etype>cause|effect)_(?P<idx>\d+)$", custom_id)
        if m:
            speech_id = m.group('sid')
            speech_narrative_id = m.group('narr')
            entity_type = m.group('etype')
            idx = m.group('idx')
            print(f"Parsed custom_id: speech_id={speech_id}, speech_narrative_id={speech_narrative_id}, entity_type={entity_type}, idx={idx}")
        else:
            m2 = re.match(r"speech_(?P<sid>[^_]+)_entity_topics_batch_(?P<b>\d+)$", custom_id)
            if m2:
                speech_id = m2.group('sid')
        # Build rows
        #print(items_list)
        for item in items_list:
            entity = item.get('entity')
            topic = item.get('topic')
            location = item.get('location')
            classification = str(item.get('classification', '')).strip().lower()
            docref = item.get('document_reference')
            row = {
                'entity': entity,
                'topic': topic,
                'classification': classification,
                'document_reference': docref,
                'custom_id': custom_id,
                'speech_id': speech_id,
                'location': location,
            }
            print(f"Extracted item: {row}")
            if idx is not None:
                row['request_idx'] = idx
            if speech_narrative_id is not None:
                row['speech_narrative_id'] = speech_narrative_id
            if entity_type is not None:
                row['entity_type'] = entity_type
            if entity and (speech_narrative_id is not None):
                row['entity_speech_narrative_id'] = f"{entity}|{speech_narrative_id}"
            df_rows.append(row)
    df = pd.DataFrame(df_rows)
    #return df
    # Optional enrichment from source CSV
    if source_csv_path and os.path.exists(source_csv_path):
        src = pd.read_csv(source_csv_path)
        if 'speech_narrative_id' not in df.columns and {'cause_entity','effect_entity','speech_narrative_id'}.issubset(src.columns):
            rows_map: Dict[str, Any] = {}
            for _, r in src.iterrows():
                narr = r.get('speech_narrative_id')
                ce = str(r.get('cause_entity')) if pd.notna(r.get('cause_entity')) else None
                ee = str(r.get('effect_entity')) if pd.notna(r.get('effect_entity')) else None
                if ce: rows_map.setdefault(ce, narr)
                if ee: rows_map.setdefault(ee, narr)
            df['speech_narrative_id'] = df['entity'].map(rows_map)
            df['entity_speech_narrative_id'] = df.apply(
                lambda r: f"{r['entity']}|{r['speech_narrative_id']}" if pd.notna(r.get('entity')) and pd.notna(r.get('speech_narrative_id')) else None,
                axis=1,
            )
        if 'entity_type' not in df.columns and {'cause_entity','effect_entity'}.issubset(src.columns):
            type_map: Dict[str, str] = {}
            for _, r in src.iterrows():
                ce = str(r.get('cause_entity')) if pd.notna(r.get('cause_entity')) else None
                ee = str(r.get('effect_entity')) if pd.notna(r.get('effect_entity')) else None
                if ce: type_map.setdefault(ce, 'cause')
                if ee: type_map.setdefault(ee, 'effect')
            df['entity_type'] = df['entity'].map(type_map)
    return df

In [13]:
json.loads('\n{\n    "items": [\n        {\n            "entity": "short-circuit the inflationary expectations dynamic",\n            "topic": "inflation",\n            "classification": "higher"\n        }\n    ]\n}\n')

{'items': [{'entity': 'short-circuit the inflationary expectations dynamic',
   'topic': 'inflation',
   'classification': 'higher'}]}

In [14]:
# Build entity or sentence batches from CSV (uses document_reference if available)
def prepare_entity_batches_from_csv(
    csv_path: str = 'batch_submissions.csv',
    batch_len: Optional[int] = None,
    pct_of_batch: float = 0.3,
) -> List[Dict[str, Any]]:
    df_src = pd.read_csv(csv_path)
    use_sentences = 'document_reference' in df_src.columns and df_src['document_reference'].notna().any()
    if use_sentences:
        sentences_series = df_src['document_reference'].dropna().astype(str).map(str.strip)
        sentences = sorted(pd.unique(sentences_series))
        if not sentences:
            return [{'speech_id': 'entities_all', 'batches': []}]
        batches = create_batches_from_sentences(sentences, batch_len=batch_len, pct_of_batch=pct_of_batch)
        return [{'speech_id': 'entities_all', 'batches': batches}]
    # Fallback to entities if no sentences available
    required_cols = {'cause_entity','effect_entity'}
    if not required_cols.issubset(df_src.columns):
        raise ValueError("CSV must contain 'document_reference' or both 'cause_entity' and 'effect_entity' columns.")
    entities_series = pd.concat([df_src['cause_entity'], df_src['effect_entity']]).dropna().astype(str).map(str.strip)
    entities = sorted(pd.unique(entities_series))
    if not entities:
        return [{'speech_id': 'entities_all', 'batches': []}]
    batches = create_batches_from_sentences(entities, batch_len=batch_len, pct_of_batch=pct_of_batch)
    return [{'speech_id': 'entities_all', 'batches': batches}]

In [16]:
pd.read_csv('Catherine_man_redone.csv')

,cause_entity,effect_entity,relationship,document_reference,quantifications,modulator_uncertainty,modulator_temporal,tense,custom_id,speech_id,batch_id,speech_narrative_id
0,recent economic shocks,uncertainties around them,recent economic shocks create uncertainties,Catherine L Mann talks about the impact of rec...,NaN,NaN,NaN,PRESENT,speech_16649_batch_0,16649,batch_695a66a3679081909dffa72c9555fbdd,16649_0
1,near-term inflationary pressures,becoming embedded in domestic expectations and...,near-term inflationary pressures embed in expe...,"As I see it, the key balance of uncertainties ...",NaN,NaN,NaN,PRESENT,speech_16649_batch_0,16649,batch_695a66a3679081909dffa72c9555fbdd,16649_1
2,projected dramatic deterioration in real purch...,deterioration in real purchasing power of peop...,deterioration in purchasing power affects incomes,"As I see it, the key balance of uncertainties ...",NaN,NaN,near and medium term,FUTURE,speech_16649_batch_0,16649,batch_695a66a3679081909dffa72c9555fbdd,16649_2
3,monetary policy tools,short-circuit the inflationary expectations dy...,monetary policy tools can short-circuit inflat...,The challenge is to deploy monetary policy too...,NaN,NaN,now,PRESENT,speech_16649_batch_0,16649,batch_695a66a3679081909dffa72c9555fbdd,16649_3
4,monetary policy tools,prevent inflation from remaining above target,monetary policy tools prevent inflation from r...,The challenge is to deploy monetary policy too...,NaN,NaN,for longer,FUTURE,speech_16649_batch_0,16649,batch_695a66a3679081909dffa72c9555fbdd,16649_4
...,...,...,...,...,...,...,...,...,...,...,...,...
407,carbon price shock,affects green sector via spillover effects,Changes in carbon pricing indirectly impact th...,It is important to consider that the carbon pr...,NaN,NaN,NaN,NaN,speech_16655_batch_3,16655,batch_695a66ac6338819096be86723ff47895,16655_100
408,carbon prices,raise the cost of capital of emissions-intensi...,Higher carbon prices increase operational cost...,Hengge et al. (2023) also find that carbon pri...,NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_695a66ac6338819096be86723ff47895,16655_101
409,contractionary monetary policy shock,output declines,Tightening monetary policy reduces economic ac...,"After a contractionary monetary policy shock, ...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_695a66ac6338819096be86723ff47895,16655_102
410,output declines,reduces the demand for emissions,"Lower output decreases industrial activity, re...","After a contractionary monetary policy shock, ...",NaN,NaN,NaN,PAST,speech_16655_batch_3,16655,batch_695a66ac6338819096be86723ff47895,16655_103


In [17]:
# Submit entity-topic classification directly from a CSV
source_csv_path = "Catherine_man_redone.csv"#'Output/csvs/Dissagregated_csvs/Catherine_man.csv'  # update to your CSV path
entity_topics_job = create_speech_batch_and_submit_entity_topics(
    source_csv_path=source_csv_path,
    topics=TOPICS,
    model='gpt-4o',
    temperature=0.0,
    completion_window='24h',
    metadata={'note': 'entity_topics_from_csv'},
)
entity_topics_job

Batch created. id=batch_695a7de2ad6881908184f18c9cba9c29 status=validating input_file_id=file-RzR5oHr3p98EPcfpbKRAik completion_window=24h


{'batch_id': 'batch_695a7de2ad6881908184f18c9cba9c29',
 'status': 'validating',
 'input_file_id': 'file-RzR5oHr3p98EPcfpbKRAik',
 'jsonl_path': 'user_outputs/speech_runs/entities_from_csv.jsonl',
 'completion_window': '24h',
 'metadata': {'mode': 'entity_topics',
  'source_csv_path': 'Catherine_man_redone.csv',
  'note': 'entity_topics_from_csv'}}

In [18]:
entity_topics_job

{'batch_id': 'batch_695a7de2ad6881908184f18c9cba9c29',
 'status': 'validating',
 'input_file_id': 'file-RzR5oHr3p98EPcfpbKRAik',
 'jsonl_path': 'user_outputs/speech_runs/entities_from_csv.jsonl',
 'completion_window': '24h',
 'metadata': {'mode': 'entity_topics',
  'source_csv_path': 'Catherine_man_redone.csv',
  'note': 'entity_topics_from_csv'}}

In [28]:
completed_entities_df = retrieve_batch_output_entity_topics(entity_topics_job['batch_id'])

{'id': 'resp_0b0f5396c0827c3600695a7e5085e081a0949fe67737213f5c', 'object': 'response', 'created_at': 1767538256, 'status': 'completed', 'background': False, 'billing': {'payer': 'developer'}, 'completed_at': 1767538257, 'error': None, 'incomplete_details': None, 'instructions': None, 'max_output_tokens': None, 'max_tool_calls': None, 'model': 'gpt-4o-2024-08-06', 'output': [{'id': 'msg_0b0f5396c0827c3600695a7e51338881a083ee69ffd90da2f2', 'type': 'message', 'status': 'completed', 'content': [{'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': '```json\n{\n    "items": [\n        {\n            "entity": "recent economic shocks",\n            "topic": "geopolitical shocks",\n            "classification": "higher",\n            "location": "global"\n        }\n    ]\n}\n```'}], 'role': 'assistant'}], 'parallel_tool_calls': True, 'previous_response_id': None, 'prompt_cache_key': None, 'prompt_cache_retention': None, 'reasoning': {'effort': None, 'summary': None}, 'safety_id

In [30]:
completed_entities_df[['topic','classification','location']].value_counts()

topic                classification  location
inflation            higher          domestic    178
output               lower           domestic     63
interest rates       higher          domestic     61
financial markets    higher          domestic     47
inflation            normal          domestic     34
interest rates       lower           domestic     30
                     normal          domestic     29
other                higher          domestic     28
interest rates       higher          global       28
other                higher          global       26
output               higher          domestic     24
inflation            lower           domestic     24
financial markets    normal          domestic     19
output               normal          domestic     19
financial markets    lower           domestic     17
geopolitical shocks  higher          global       16
inflation            higher          global       14
labour market        higher          domestic     12


In [31]:
completed_entities_df.to_csv('Entity_Topic_Classifications_Catherine_man_detailed.csv', index=False)
#[['topic','classification']].value_counts()

In [126]:
for i in completed_entities_df:
    print(i.get('content',[]))
    g = i.get('content',[])
    print([j.get('text',str) for j in g])

[{'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': '{"items":[{"entity":"monetary policy","topic":"interest rates","classification":"higher"}]}'}]
['{"items":[{"entity":"monetary policy","topic":"interest rates","classification":"higher"}]}']


In [130]:
# Preview merged results with source metadata
completed_entities_df#.head()

""


In [ ]:
# Retrieve and inspect results from the CSV-based batch
df_entity_topics = retrieve_batch_output_entity_topics(entity_topics_job['batch_id'], source_csv_path=source_csv_path)
df_entity_topics.head()

In [ ]:
# Quick test: verify custom_id parsing works as expected
_test_id = 'speech_16655_narr_16655_44_cause_293'
m = re.match(r"speech_(?P<sid>[^_]+)_narr_(?P<narr>.*?)_(?P<etype>cause|effect)_(?P<idx>\d+)$", _test_id)
print({'matched': bool(m), 'sid': m.group('sid') if m else None, 'narr': m.group('narr') if m else None, 'etype': m.group('etype') if m else None, 'idx': m.group('idx') if m else None})